# Medicina Democrática Studio no Android

Abra no navegador, conecte a um ambiente **CPU gratuito** e execute as duas células. A primeira prepara as dependências uma vez por sessão.

No [Google AI Studio](https://aistudio.google.com/api-keys), use uma chave de um projeto marcado **Free Tier**, com faturamento desativado. No Colab, abra **Segredos** (🔑), adicione o nome GEMINI_API_KEY e ative o acesso deste notebook. O valor fica no painel de segredos.

Na segunda célula, cole o roteiro integral, confirme a aprovação e toque em **Produzir episódio**. Baixe o MP4 e, se quiser, a legenda SRT enquanto a sessão estiver conectada. A cota gratuita do Google e o tempo do Colab dependem da disponibilidade da conta.

In [ ]:
# @title 1. Preparar o estúdio
import os
import subprocess
from pathlib import Path

repo = Path("/content/medicina-democratica-studio")
runtime = Path("/content/md-studio-runtime")
safe_env = {name: value for name, value in os.environ.items()
            if name not in {"GEMINI_API_KEY", "GEMINI_API_KEY_FILE"}}
if not repo.exists():
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", "main",
        "https://github.com/georgesmarcel2014-ui/medicina-democratica-studio.git",
        str(repo),
    ], check=True, env=safe_env)

subprocess.run(["bash", "deploy/setup-standalone.sh", str(runtime)],
               cwd=repo, check=True, env=safe_env)
print("Estúdio pronto. Execute a próxima célula.")

In [ ]:
# @title 2. Produzir um episódio
import json
import tempfile
import ipywidgets as widgets
from google.colab import files, userdata
from IPython.display import display

jobs = runtime / "episodios"
jobs.mkdir(mode=0o700, exist_ok=True)
field = widgets.Layout(width="100%")
titulo = widgets.Text(placeholder="Título do episódio", layout=field)
serie = widgets.Dropdown(options=[("Na Prática", "na-pratica"), ("Notícias", "noticias"),
                                  ("História", "historia")], description="Série")
roteiro = widgets.Textarea(placeholder="Cole aqui o roteiro aprovado", rows=8, layout=field)
aprovado = widgets.Checkbox(description="Aprovo o roteiro para narração", indent=False, layout=field)
gratuito = widgets.Checkbox(description="Projeto Free Tier, sem faturamento", indent=False, layout=field)
produzir = widgets.Button(description="Produzir episódio", button_style="primary")
baixar_mp4 = widgets.Button(description="Baixar MP4", disabled=True)
baixar_srt = widgets.Button(description="Baixar SRT", disabled=True)
mensagens = widgets.Output()
trabalho = {}

def executar(_):
    with mensagens:
        mensagens.clear_output(wait=True)
        nome, texto = titulo.value.strip(), roteiro.value.strip()
        if not (0 < len(nome) <= 140 and 0 < len(texto) <= 15_000 and aprovado.value and gratuito.value):
            print("Preencha título e roteiro e marque as duas confirmações.")
            return
        ordem = (nome, serie.value, texto)
        if trabalho.get("ordem") != ordem:
            trabalho.clear()
            trabalho.update(ordem=ordem, pasta=Path(tempfile.mkdtemp(dir=jobs)))
        pasta = trabalho["pasta"]
        video = pasta / "Medicina_Democratica.mp4"
        srt = video.with_suffix(".srt")
        if trabalho.get("mp4"):
            print("Episódio pronto para download.")
            return

        produzir.disabled = baixar_mp4.disabled = baixar_srt.disabled = True
        try:
            env = dict(safe_env)
            env.update(
                PATH=f"{runtime}/node/bin:{env['PATH']}",
                STUDIO_ASR_PYTHON=str(runtime / "asr/bin/python"),
                STUDIO_WHISPER_MODEL=str(runtime / "whisper"),
                CHROME_PATH=(runtime / "chrome-path").read_text().strip(),
            )
            # Um WAV concluído para esta mesma ordem pode ser reaproveitado se o render falhar.
            audio = {"path": "narration.wav", "source": "google-ai-studio", "approved": True}
            if not (pasta / "narration.wav").is_file():
                try:
                    env["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
                except (userdata.SecretNotFoundError, userdata.NotebookAccessError, userdata.TimeoutException):
                    print("Configure GEMINI_API_KEY no painel Segredos e autorize este notebook.")
                    return
                audio = {"path": "narration.wav", "source": "google-tts", "scriptApproved": True}

            entrada = pasta / "episode.json"
            entrada.write_text(json.dumps(dict(title=nome, series=serie.value, script=texto,
                                               visualIdentity="medicina-democratica", audio=audio),
                                          ensure_ascii=False), encoding="utf-8")
            print("Produzindo um episódio. Mantenha a sessão conectada.", flush=True)
            resultado = subprocess.run([
                "node", "--import", "tsx", "src/pipeline/cli.ts", "produce", "--free-tier",
                "--input", str(entrada), "--output", str(video),
            ], cwd=repo, env=env, text=True, capture_output=True)
            env.pop("GEMINI_API_KEY", None)
            if resultado.returncode:
                avisos = {
                    "TTS_ACCESS_DENIED": "O Google recusou o acesso. Confira a chave e o projeto Free Tier.",
                    "TTS_RATE_LIMITED": "O Google atingiu a cota. Aguarde a renovação da cota gratuita.",
                    "TTS_FAILED": "O Google TTS não concluiu a narração. Confira o acesso ao modelo no AI Studio.",
                    "NARRATION_TOO_LONG": "A narração passou de 179 segundos. Aprove um roteiro mais curto.",
                    "CAPTIONS_MISMATCH": "A fala divergiu do roteiro. Revise a narração antes de continuar.",
                    "CAPTIONS_FAILED": "O alinhamento das legendas falhou. Execute a preparação novamente.",
                }
                print(avisos.get(resultado.stderr.strip(), "A produção falhou. Verifique a preparação e a conexão."))
                return

            Path(f"{video}.srt").replace(srt)
            relatorio = json.loads(Path(f"{video}.report.json").read_text())
            trabalho.update(mp4=video, srt=srt)
            baixar_mp4.disabled = baixar_srt.disabled = False
            print(f"Pronto. {relatorio['width']} × {relatorio['height']} · {relatorio['duration']:.1f} s.")
        finally:
            produzir.disabled = False

produzir.on_click(executar)
baixar_mp4.on_click(lambda _: files.download(str(trabalho["mp4"])))
baixar_srt.on_click(lambda _: files.download(str(trabalho["srt"])))
display(widgets.VBox([titulo, serie, roteiro, aprovado, gratuito, produzir,
                      widgets.HBox([baixar_mp4, baixar_srt]), mensagens]))